In [1]:
import cv2
import os
import matplotlib.pyplot as plt

In [2]:
ROOT_DIR = "C:\\Adrianov\\Projects\\Project-Satanael\\"
DEMO_DIR = os.path.join(ROOT_DIR, 'attack_demo')
APRICOT_TEST_DIR = os.path.join(ROOT_DIR, 'data', 'APRICOTv1.0', 'Images', 'Test')
# TJUDHD_TEST_DIR = os.path.join(ROOT_DIR, 'data', 'tju-dhd', 'patched_random', 'images')
# TJUDHD_LABEL_DIR = os.path.join(ROOT_DIR, 'data', 'tju-dhd', 'patched_random', 'labels')
TJUDHD_TRAIN_DIR = os.path.join(ROOT_DIR, 'data', 'tju-dhd', 'patched_random_bigger', 'images')
TJUDHD_TRAIN_LABEL_DIR = os.path.join(ROOT_DIR, 'data', 'tju-dhd', 'patched_random_bigger', 'labels')
TJUDHD_VAL_DIR = os.path.join(ROOT_DIR, 'data', 'tju-dhd', 'patched_random_bigger_val', 'images')
TJUDHD_VAL_LABEL_DIR = os.path.join(ROOT_DIR, 'data', 'tju-dhd', 'patched_random_bigger_val', 'labels')
TJUDHD_TEST_DIR = os.path.join(ROOT_DIR, 'data', 'tju-dhd', 'patched_random_bigger_test', 'images')
TJUDHD_TEST_LABEL_DIR = os.path.join(ROOT_DIR, 'data', 'tju-dhd', 'patched_random_bigger_test', 'labels')
SEG_RES_DIR = os.path.join(ROOT_DIR, 'results_segment_grey')
MASK_RES_DIR = os.path.join(ROOT_DIR, 'results_mask')
MASK_RES_DIR_CD_APRICOT = os.path.join(ROOT_DIR, 'results_cd_grey_test_APRICOT')
MASK_RES_DIR_ADAPTIVE = os.path.join(ROOT_DIR, 'results_mask_adaptive')
MASK_RES_DIR_CD = os.path.join(ROOT_DIR, 'results_cd_grey')


TJUDHD_TEST_DIR = os.path.join(ROOT_DIR, 'data', 'tju-dhd', 'patched_random_bigger_test', 'images')
TJUDHD_TEST_LABEL_DIR = os.path.join(ROOT_DIR, 'data', 'tju-dhd', 'patched_random_bigger_test', 'labels')
MASK_RES_DIR_PAD = os.path.join(ROOT_DIR, 'pad_res')

## Calculate Recall value of Segmentation

In [3]:
import json
import numpy as np
import cv2
import skimage.measure as skms


def get_regions_from_mask(mask, min_area=0):
    """Extract connected regions from a binary mask."""
    label = skms.label(mask)
    props = skms.regionprops(label)
    
    regions = []
    for i, prop in enumerate(props):
        if prop.area >= min_area:
            region_mask = (label == i + 1).astype(np.uint8)
            regions.append({
                'mask': region_mask,
                'bbox': prop.bbox,
                'area': prop.area
            })
    return regions

In [4]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

def calculate_iou(region_mask, bbox, bbox_format="coco", epsilon=1e-8, debug=False):
    """
    Compute IoU between a region mask and a bounding box.
    
    Args:
        region_mask (np.ndarray): Binary mask of region (H x W).
        bbox (list/tuple): Bounding box coordinates.
            - COCO format: [x, y, w, h]
            - YOLO format: [x_center, y_center, w, h] (normalized to [0,1])
        bbox_format (str): "coco" or "yolo".
        epsilon (float): Small constant to avoid division by zero.
        debug (bool): If True, visualize the region mask, bbox, and overlaps.
    """
    H, W = region_mask.shape
    
    if bbox_format == "coco":
        x1, y1, w, h = bbox
        x1, y1, x2, y2 = int(x1), int(y1), int(x1 + w), int(y1 + h)

    elif bbox_format == "yolo":
        # YOLO format is normalized: [x_center, y_center, w, h]
        x_center, y_center, w, h = bbox
        x_center, y_center, w, h = x_center * W, y_center * H, w * W, h * H
        x1 = int(x_center - w / 2)
        y1 = int(y_center - h / 2)
        x2 = int(x_center + w / 2)
        y2 = int(y_center + h / 2)

    else:
        raise ValueError("bbox_format must be either 'coco' or 'yolo'")

    # Clip to image boundaries
    if bbox_format == 'yolo':
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(W, x2), min(H, y2)

    bbox_mask = np.zeros_like(region_mask, dtype=np.uint8)
    bbox_mask[y1:y2, x1:x2] = 1

    intersection = np.logical_and(region_mask, bbox_mask).sum()
    union = np.logical_or(region_mask, bbox_mask).sum()
    iou = intersection / (union + epsilon)

    if debug:
        # Prepare visualization
        vis = np.zeros((H, W, 3), dtype=np.uint8)

        # Region mask in red
        vis[region_mask.astype(bool)] = [255, 0, 0]

        # Bbox mask in green
        vis[bbox_mask.astype(bool)] = [0, 255, 0]

        # Intersection in yellow
        intersection_mask = np.logical_and(region_mask, bbox_mask)
        vis[intersection_mask] = [255, 255, 0]

        # Draw bbox rectangle outline
        vis = cv2.rectangle(vis, (x1, y1), (x2, y2), (255, 255, 255), 1)

        plt.figure(figsize=(6,6))
        plt.imshow(vis)
        plt.title(f"IoU = {iou:.4f}")
        plt.axis("off")
        plt.show()

    return iou


In [5]:
def match_region(region_mask, anns, threshold=0.5, mode='coco'):
    """Check if region matches any annotation (IoU ≥ threshold)."""
    for ann in anns:
        iou = calculate_iou(region_mask, ann['bbox'], bbox_format=mode, debug=False)
        # print(iou)
        if iou >= threshold:
            return True
    return False



In [6]:
import os

def tally_tp_yolo(label_dir, image_filename, category_ids, mask, threshold=0.5):
    regions = get_regions_from_mask(mask)
    label_path = os.path.join(label_dir, image_filename + '.txt')
    
    anns = []
    with open(label_path, "r") as f:
        i = 0
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            class_id = int(parts[0])
            if class_id in category_ids:
                x_center, y_center, w, h = map(float, parts[1:5])
                anns.append({
                    'id': i,
                    'bbox': [x_center, y_center, w, h],
                    'category_id': class_id,
                    'area': w * h
                })
                i += 1

    # If there are no ground-truth anns, return zeros (no TP / FN)
    if not anns:
        return 0, 0

    # track which ground-truth anns have been matched (True => TP)
    ann_intersected = {ann['id']: False for ann in anns}

    # small epsilon to detect any overlap (IoU > 0)
    overlap_eps = 1e-6

    fp = 0
    
    for r in regions:
        # collect all bboxes that have any overlap with this region
        intersecting_anns = []
        for ann in anns:
            # use a tiny threshold to detect any overlap
            if match_region(r['mask'], [ann], threshold=overlap_eps, mode='yolo'):
                intersecting_anns.append(ann)

        if not intersecting_anns:
            # nothing to do for this region (no GT bbox touches it)
            fp += 1
            continue

        # single intersecting bbox: test with configured threshold
        if len(intersecting_anns) == 1:
            ann = intersecting_anns[0]
            if match_region(r['mask'], [ann], threshold=threshold, mode='yolo'):
                ann_intersected[ann['id']] = True
            else:
                fp += 1 # Mask region does not satisfy threshold (is a false positive)
        else:
            # multiple intersecting bboxes -> merge and test the merged bbox
            merged_ann = merge_yolo_bboxes(intersecting_anns)
            if match_region(r['mask'], [merged_ann], threshold=threshold, mode='yolo'):
                # mark all intersecting anns as detected
                for ann in intersecting_anns:
                    ann_intersected[ann['id']] = True
            else:
                fp += 1 # Mask region does not satisfy threshold (is a false positive)

    tp = sum(1 for v in ann_intersected.values() if v)
    fn = sum(1 for v in ann_intersected.values() if not v)
    return tp, fn, fp


def merge_yolo_bboxes(anns):
    """
    Merge multiple YOLO-format bboxes into a single bbox.
    Each ann['bbox'] is [xc, yc, w, h] (same format you read from labels).
    Returns an ann dict compatible with match_region (same keys as input anns).
    """
    xs = []
    ys = []
    for ann in anns:
        xc, yc, w, h = ann['bbox']
        x1 = xc - w / 2.0
        y1 = yc - h / 2.0
        x2 = xc + w / 2.0
        y2 = yc + h / 2.0
        xs.extend([x1, x2])
        ys.extend([y1, y2])

    x1, x2 = min(xs), max(xs)
    y1, y2 = min(ys), max(ys)

    merged_w = x2 - x1
    merged_h = y2 - y1
    merged_xc = (x1 + x2) / 2.0
    merged_yc = (y1 + y2) / 2.0

    return {
        'bbox': [merged_xc, merged_yc, merged_w, merged_h],
        'category_id': anns[0]['category_id'],
        'area': merged_w * merged_h
    }


# Recall for TJU-DHD Dataset

## Recall at IoU=0.5

In [9]:
from tqdm import tqdm
import shutil

# d_types = ['Test', 'Train']
d_types = ['Test']

patch_types = [
    'Naturalistic1', 'Naturalistic2', 'Naturalistic3',
    'Naturalistic4', 'Naturalistic5', 'Naturalistic6',
    'TSEA1', 'TSEA2'
]

test_types = ['Naturalistic5', 'Naturalistic6', 'TSEA2']

for d_type in d_types:
    for patch in patch_types:
        tp = 0
        fn = 0
        if d_type == 'Train' and patch in test_types:
            continue
        if d_type == 'Train':
            label_dir = 'labels'
        if d_type == 'Test':
            label_dir = 'labels_w_adv'
            # label_dir = 'labels'

        eval_path = os.path.join(ROOT_DIR, 'data', 'tju-dhd', 'eval_final_patched', d_type, patch, 'images')
        label_path = os.path.join(ROOT_DIR, 'data', 'tju-dhd', 'eval_final_patched', d_type, patch, label_dir)
        heatmap_path = os.path.join(ROOT_DIR, 'results_mi_grey_test_final_eval')
        # heatmap_path = os.path.join(ROOT_DIR, 'results_pad_test_final_eval')

        # print(eval_path, label_path, mask_path)
        
        for fname in tqdm(os.listdir(eval_path), desc="Evaluating images"):
            if not fname.endswith('.jpg'):
                continue
            try:
                mask_path = os.path.join(heatmap_path, f'{fname.replace(".jpg", ".jpg")}_cd_thresh.png')
                # mask_path = os.path.join(heatmap_path, f'{fname.replace(".jpg", "")}_pad_mask.png')
                # print(fname)
                # print(mask_path)
                mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
                mask = (mask > 0).astype(np.uint8)
            except:
                continue
                
            annot_dir = label_path
            t_tp, t_fn, _ = tally_tp_yolo(
                label_dir=annot_dir,
                image_filename=os.path.splitext(fname)[0],
                category_ids=[1],
                mask=mask,
                threshold=0.5
            )
            if t_fn > 0:
                print(f"Undetected adversarial patch @{fname}")
                shutil.copy(mask_path, 'false_negs_MI')
            tp += t_tp
            fn += t_fn
        
        print(tp)
        print(fn)
        print(f"Final recall score (patch type {patch} {d_type}) for segmentation method: {(tp/(tp+fn))*100}%")

Evaluating images:   4%|██▎                                                      | 2/50 [00:00<00:09,  4.95it/s]

Undetected adversarial patch @Naturalistic1_1499917703635.jpg
Undetected adversarial patch @Naturalistic1_1499918339461.jpg


Evaluating images:  10%|█████▋                                                   | 5/50 [00:01<00:08,  5.07it/s]

Undetected adversarial patch @Naturalistic1_1499925371739.jpg
Undetected adversarial patch @Naturalistic1_1499931818829.jpg


Evaluating images:  16%|█████████                                                | 8/50 [00:01<00:08,  4.68it/s]

...

Undetected adversarial patch @TSEA2_1497343727766.jpg


Evaluating images:  99%|███████████████████████████████████████████████████████▎| 79/80 [00:22<00:00,  3.33it/s]

Undetected adversarial patch @TSEA2_1497344149451.jpg


Evaluating images: 100%|████████████████████████████████████████████████████████| 80/80 [00:23<00:00,  3.45it/s]

56
53
Final recall score (patch type TSEA2 Test) for segmentation method: 51.37614678899083%


In [ ]:
import os
import cv2
from tqdm import tqdm
import matplotlib.pyplot as plt

# Assumes ROOT_DIR and tally_tp_yolo are defined elsewhere

d_type = 'Test'  # choose 'Test' (matches your previous usage). If you want both, loop over d_types.
# patch grouping (unchanged)
patch_types = [
    ['Naturalistic1', 'Naturalistic2', 'Naturalistic3',
     'Naturalistic4', 'Naturalistic5', 'Naturalistic6'],
    ['TSEA1', 'TSEA2']
]

# thresholds requested: 45..95 inclusive
thresholds = list(range(20, 96))

recall = [[], []]

heatmap_path = os.path.join(ROOT_DIR, 'results_mi_grey_test_final_eval')  # reuse outside loop

for thr in tqdm(thresholds, desc="Evaluating across thresholds..."):

    for j in range(2):
        tp = 0
        fn = 0
        patches = patch_types[j]
        for patch in patches:
            # choose label directory based on d_type
            if d_type == 'Train':
                label_dir = 'labels'
            else:
                label_dir = 'labels_w_adv'

            eval_path = os.path.join(ROOT_DIR, 'data', 'tju-dhd', 'eval_final_patched', d_type, patch, 'images')
            label_path = os.path.join(ROOT_DIR, 'data', 'tju-dhd', 'eval_final_patched', d_type, patch, label_dir)

            # Defensive: skip if eval_path doesn't exist
            if not os.path.isdir(eval_path):
                continue

            for fname in os.listdir(eval_path):
                if not fname.lower().endswith('.jpg'):
                    continue
                # Build mask path using the base filename and suffix _cd_thresh.png
                mask_path = os.path.join(heatmap_path, f'{fname.replace(".jpg", ".jpg")}_cd_thresh.png')

                if not os.path.isfile(mask_path):
                    # skip if predicted mask doesn't exist
                    continue

                mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
                if mask is None:
                    continue
                mask = (mask > 0).astype('uint8')

                t_tp, t_fn, _ = tally_tp_yolo(
                    label_dir=label_path,
                    image_filename=os.path.splitext(fname)[0],
                    category_ids=[1],
                    mask=mask,
                    threshold=thr/100.0
                )
                tp += t_tp
                fn += t_fn

        # Avoid division by zero when there are no positive annotations in the evaluated set
        denom = (tp + fn)
        if denom == 0:
            recall_value = 0.0
        else:
            recall_value = (tp / denom) * 100.0

        # Fix from your original: changed trailing brace to closing parenthesis
        recall[j].append(recall_value)

# Plotting
plt.figure(figsize=(8, 5))
plt.plot(thresholds, recall[0], label='Naturalistic', color='orange', marker='o')
plt.plot(thresholds, recall[1], label='T-SEA', color='blue', marker='o')
plt.xlabel('Threshold (%)')
plt.ylabel('Recall (%)')
plt.title('Recall vs Threshold')
plt.xlim(min(thresholds), max(thresholds))
plt.ylim(0, 100)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()


Evaluating across thresholds...:   3%|█                                       | 2/76 [05:12<3:10:37, 154.57s/it]

## Overall Recall, Precision, F1

In [40]:
from tqdm import tqdm


# d_types = ['Test', 'Train']
d_types = ['Test']

patch_types = [
    'Naturalistic1', 'Naturalistic2', 'Naturalistic3',
    'Naturalistic4', 'Naturalistic5', 'Naturalistic6',
    'TSEA1', 'TSEA2'
]

test_types = ['Naturalistic5', 'Naturalistic6', 'TSEA2']

for d_type in d_types:
    tp = 0
    fn = 0
    fp = 0
    for patch in patch_types:
        if d_type == 'Train' and patch in test_types:
            continue
        if d_type == 'Train':
            label_dir = 'labels'
        if d_type == 'Test':
            label_dir = 'labels_w_adv'

        eval_path = os.path.join(ROOT_DIR, 'data', 'tju-dhd', 'eval_final_patched', d_type, patch, 'images')
        label_path = os.path.join(ROOT_DIR, 'data', 'tju-dhd', 'eval_final_patched', d_type, patch, label_dir)
        heatmap_path = os.path.join(ROOT_DIR, 'results', 'adv_mask')
        # heatmap_path = os.path.join(ROOT_DIR, 'results_pad_test_final_eval')

        # print(eval_path, label_path, mask_path)
        
        for fname in tqdm(os.listdir(eval_path), desc="Evaluating images"):
            if not fname.endswith('.jpg'):
                continue
            try:
                mask_path = os.path.join(heatmap_path, f'{fname.replace(".jpg", "")}_mask_xgb.png')
                # mask_path = os.path.join(heatmap_path, f'{fname.replace(".jpg", "")}_pad_mask.png')
                # print(fname)
                # print(mask_path)
                mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
                mask = (mask > 0).astype(np.uint8)
            except:
                continue
                
            annot_dir = label_path
            t_tp, t_fn, t_fp = tally_tp_yolo(
                label_dir=annot_dir,
                image_filename=os.path.splitext(fname)[0],
                category_ids=[1],
                mask=mask,
                threshold=0.5
            )
            tp += t_tp
            fn += t_fn
            fp += t_fp
        
            # print(tp)
            # print(fn)
    recall = (tp/(tp+fn))*100
    precision = (tp/(tp+fp))*100
    f1 = (2*precision*recall)/(precision + recall)
    print(f"Final recall score (Overall Test Set) for Ours (XGB): {recall}%")
    print(f"Final precision score (Overall Test Set) for Ours (XGB): {precision}%")
    print(f"Final F1 score (Overall Test Set) for Ours (XGB): {f1}%")

 ... (more hidden) ...
 ... (more hidden) ...
 ... (more hidden) ...
 ... (more hidden) ...
 ... (more hidden) ...
 ... (more hidden) ...
 ... (more hidden) ...
 ... (more hidden) ...

Final recall score (Overall Test Set) for Ours (XGB): 88.35403726708074%
Final precision score (Overall Test Set) for Ours (XGB): 97.59862778730704%
Final F1 score (Overall Test Set) for Ours (XGB): 92.74653626731867%


### Feature Extraction

### Feature Extraction for TJUDHD

In [13]:
import os

ROOT_DIR = "C:\\Adrianov\\Projects\\Project-Satanael\\"
MASK_RES_DIR = os.path.join(ROOT_DIR, 'results_cd_grey_test_final_eval')

DATA_DIR = os.path.join(ROOT_DIR, 'data', 'tju-dhd', 'eval_final_patched')

In [14]:
import sys
import importlib.util
import warnings

color_path = os.path.abspath("../defenselib/feature_extraction/color.py")

spec = importlib.util.spec_from_file_location("color_ext", color_path)
color_ext = importlib.util.module_from_spec(spec)
sys.modules["color_ext"] = color_ext
spec.loader.exec_module(color_ext)


In [15]:
import sys
import importlib.util
import warnings

texture_path = os.path.abspath("../defenselib/feature_extraction/texture.py")

spec = importlib.util.spec_from_file_location("texture_ext", texture_path)
texture_ext = importlib.util.module_from_spec(spec)
sys.modules["texture_ext"] = texture_ext
spec.loader.exec_module(texture_ext)


In [16]:
import json
import numpy as np
import cv2
import skimage.measure as skms


def calculate_iou(region_mask, bbox, bbox_format="coco", epsilon=1e-8, debug=False):
    """
    Compute IoU between a region mask and a bounding box.
    
    Args:
        region_mask (np.ndarray): Binary mask of region (H x W).
        bbox (list/tuple): Bounding box coordinates.
            - COCO format: [x, y, w, h]
            - YOLO format: [x_center, y_center, w, h] (normalized to [0,1])
        bbox_format (str): "coco" or "yolo".
        epsilon (float): Small constant to avoid division by zero.
        debug (bool): If True, visualize the region mask, bbox, and overlaps.
    """
    H, W = region_mask.shape
    
    if bbox_format == "coco":
        x1, y1, w, h = bbox
        x1, y1, x2, y2 = int(x1), int(y1), int(x1 + w), int(y1 + h)

    elif bbox_format == "yolo":
        # YOLO format is normalized: [x_center, y_center, w, h]
        x_center, y_center, w, h = bbox
        x_center, y_center, w, h = x_center * W, y_center * H, w * W, h * H
        x1 = int(x_center - w / 2)
        y1 = int(y_center - h / 2)
        x2 = int(x_center + w / 2)
        y2 = int(y_center + h / 2)

    else:
        raise ValueError("bbox_format must be either 'coco' or 'yolo'")

    # Clip to image boundaries
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(W, x2), min(H, y2)

    bbox_mask = np.zeros_like(region_mask, dtype=np.uint8)
    bbox_mask[y1:y2, x1:x2] = 1

    intersection = np.logical_and(region_mask, bbox_mask).sum()
    union = np.logical_or(region_mask, bbox_mask).sum()
    iou = intersection / (union + epsilon)

    if debug:
        # Prepare visualization
        vis = np.zeros((H, W, 3), dtype=np.uint8)
    
        # Region mask in red
        vis[region_mask.astype(bool)] = [255, 0, 0]

        # Bbox mask in green
        vis[bbox_mask.astype(bool)] = [0, 255, 0]

        # Intersection in yellow
        intersection_mask = np.logical_and(region_mask, bbox_mask)
        vis[intersection_mask] = [255, 255, 0]

        # Draw bbox rectangle outline
        vis = cv2.rectangle(vis, (x1, y1), (x2, y2), (255, 255, 255), 1)

        plt.figure(figsize=(6,6))
        plt.imshow(vis)
        plt.title(f"IoU = {iou:.4f}")
        plt.axis("off")
        plt.show()

    return iou


def get_regions_from_mask(mask, min_area=60):
    """Extract connected regions from a binary mask."""
    label = skms.label(mask)
    props = skms.regionprops(label)
    
    regions = []
    for i, prop in enumerate(props):
        if prop.area >= min_area:
            region_mask = (label == i + 1).astype(np.uint8)
            regions.append({
                'mask': region_mask,
                'bbox': prop.bbox,
                'area': prop.area
            })
    return regions

def match_region(region_mask, anns, threshold=0.5, mode='coco'):
    """Check if region matches any annotation (IoU ≥ threshold)."""
    for ann in anns:
        iou = calculate_iou(region_mask, ann['bbox'], bbox_format=mode, debug=False)
        # print(iou)
        if iou >= threshold:
            return True
    return False

def merge_yolo_bboxes(anns):
    """
    Merge multiple YOLO-format bboxes into a single bbox.
    Each ann['bbox'] is [xc, yc, w, h] (same format you read from labels).
    Returns an ann dict compatible with match_region (same keys as input anns).
    """
    xs = []
    ys = []
    for ann in anns:
        xc, yc, w, h = ann['bbox']
        x1 = xc - w / 2.0
        y1 = yc - h / 2.0
        x2 = xc + w / 2.0
        y2 = yc + h / 2.0
        xs.extend([x1, x2])
        ys.extend([y1, y2])

    x1, x2 = min(xs), max(xs)
    y1, y2 = min(ys), max(ys)

    merged_w = x2 - x1
    merged_h = y2 - y1
    merged_xc = (x1 + x2) / 2.0
    merged_yc = (y1 + y2) / 2.0

    return {
        'bbox': [merged_xc, merged_yc, merged_w, merged_h],
        'category_id': anns[0]['category_id'],
        'area': merged_w * merged_h
    }


In [17]:
import skimage.measure as skms
from tqdm import tqdm
import pandas as pd
import time

BINS = 32
THRESHOLD = 0.5
RESIZE = 1024
COL_BINS = True
COL_MOMENTS = True
COL_GLCM = True
COL_GABOR = False
DISTS = [1,2,4,8,16,32,64]
ANGLES = [0, np.pi/4, np.pi/2, 3*np.pi/4]
GABOR_FREQS = [0.1, 0.2, 0.3, 0.4]

D_TYPES = ['Test', 'Train']

PATCH_TYPES = [
    'Naturalistic1', 'Naturalistic2', 'Naturalistic3',
    'Naturalistic4', 'Naturalistic5', 'Naturalistic6',
    'TSEA1', 'TSEA2'
]

TEST_TYPES = ['Naturalistic5', 'Naturalistic6', 'TSEA2']

col_names = ["image_name", "region_id", "area"]

if COL_BINS:
    col_names += [f"rgb_R_bin{i}" for i in range(BINS)] + \
                [f"rgb_G_bin{i}" for i in range(BINS)] + \
                [f"rgb_B_bin{i}" for i in range(BINS)] + \
                [f"hsv_H_bin{i}" for i in range(BINS)] + \
                [f"hsv_S_bin{i}" for i in range(BINS)] + \
                [f"hsv_V_bin{i}" for i in range(BINS)]
if COL_MOMENTS:
    col_names += (
        [f"rgb_R_{stat}" for stat in ["mean", "std", "skew"]] +
        [f"rgb_G_{stat}" for stat in ["mean", "std", "skew"]] +
        [f"rgb_B_{stat}" for stat in ["mean", "std", "skew"]] +
        [f"hsv_H_{stat}" for stat in ["mean", "std", "skew"]] +
        [f"hsv_S_{stat}" for stat in ["mean", "std", "skew"]] +
        [f"hsv_V_{stat}" for stat in ["mean", "std", "skew"]]
    )

glcm_extended = False
adv_extended = False
gabor_extended = False

start = time.time()

for d_type in D_TYPES:
    df_features = pd.DataFrame(columns=col_names)
    for patch in PATCH_TYPES:
        if d_type == 'Train' and patch in TEST_TYPES:
            continue
        if d_type == 'Train':
            label_dir = 'labels'
        if d_type == 'Test':
            label_dir = 'labels_w_adv'
        annot_dir = os.path.join(DATA_DIR, d_type, patch, label_dir)
        data_dir = os.path.join(DATA_DIR, d_type, patch, 'images')

        i = 0
        for fname in tqdm(os.listdir(data_dir), desc=f"Extracting features {patch} {d_type}..."):
        
            if not fname.endswith('.jpg'):
                continue
        
            image = cv2.imread(os.path.join(data_dir, fname))
            
            h, w = image.shape[:2]
            
            if h < w:
                new_h = RESIZE
                new_w = int(w * (RESIZE / h))
                scale_x = new_w / w
                scale_y = new_h / h
            else:
                new_w = RESIZE
                new_h = int(h * (RESIZE / w))
                scale_x = new_w / w
                scale_y = new_h / h
            
            # --- Resize image ---
            image = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_AREA)
            
            # --- Resize mask ---
            mask_path = os.path.join(MASK_RES_DIR, f'{fname}_cd_thresh.png')
            # print(mask_path)
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            mask = cv2.resize(mask, (new_w, new_h), interpolation=cv2.INTER_NEAREST)
            mask = (mask > 0).astype(np.uint8)
            
            # --- Find ann for this image ---
            label_path = os.path.join(annot_dir, fname.replace(".jpg",".txt"))
            anns = []
            
            category_ids = [1] # Adversarial patch
            
            with open(label_path, "r") as f:
                idx = 0
                for line in f:
                    parts = line.strip().split()
                    if not parts:
                        continue
                    class_id = int(parts[0])
                    if class_id in category_ids:
                        x_center, y_center, w, h = map(float, parts[1:5])
                        anns.append({
                            'id': idx,
                            'bbox': [x_center, y_center, w, h],
                            'category_id': class_id,
                            'area': w * h
                        })
                        idx += 1
                                
            regions = get_regions_from_mask(mask)
                    
            # print(anns) 
            ann_intersected = {ann['id']: False for ann in anns}
            # small epsilon to detect any overlap (IoU > 0)
            overlap_eps = 1e-6
            
            for r in regions:
                intersecting_anns = []
                for ann in anns:
                    # use a tiny threshold to detect any overlap
                    if match_region(r['mask'], [ann], threshold=overlap_eps, mode='yolo'):
                        intersecting_anns.append(ann)
        
                if not intersecting_anns:
                    # nothing to do for this region (no GT bbox touches it)
                    r['adversarial'] = False
                    continue
        
                # single intersecting bbox: test with configured threshold
                if len(intersecting_anns) == 1:
                    ann = intersecting_anns[0]
                    if match_region(r['mask'], [ann], threshold=THRESHOLD, mode='yolo'):
                        r['adversarial'] = True
                    else:
                        r['adversarial'] = False
                    # else: leave as False (no match)
                else:
                    # multiple intersecting bboxes -> merge and test the merged bbox
                    merged_ann = merge_yolo_bboxes(intersecting_anns)
                    if match_region(r['mask'], [merged_ann], threshold=THRESHOLD, mode='yolo'):
                        # mark all intersecting anns as detected
                        r['adversarial'] = True
                    else:
                        r['adversarial'] = False
                    # else: leave them as False (no match)
        
                                    
            feature_list = []
            for region_id, r in enumerate(regions):
                feats = [fname, region_id + 1, r['area']]
            
                if COL_BINS:
                    # print("AAAA", image.shape)
                    # print("BBBB", r['mask'].shape)
                    feats.extend(color_ext.extract_color_histograms(image, r['mask'], BINS))
            
                if COL_MOMENTS:
                    feats.extend(color_ext.extract_color_moments(image, r['mask']))  
        
                if COL_GLCM:
                    haralick = texture_ext.extract_haralick_features(image, r['mask'], DISTS, ANGLES)
                    feats.extend(list(haralick.values()))
                    if not glcm_extended:
                        col_names.extend(list(haralick.keys()))
                        glcm_extended = True
                        
                if COL_GABOR:
                    gabor_feats = texture_ext.extract_gabor_features(image, r['mask'], GABOR_FREQS, ANGLES)
                    feats.extend(list(gabor_feats.values()))
                    if not gabor_extended:
                        col_names.extend(list(gabor_feats.keys()))
                        gabor_extended = True
                        
                feats.append(r['adversarial'])
                feature_list.append(feats)
        
            if not adv_extended:
                col_names += ["adversarial"]
                adv_extended = True
            # print(col_names)
            df = pd.DataFrame(feature_list, columns=col_names)
            df_features = pd.concat([df_features, df], ignore_index=True)
            i += 1

        if d_type == 'Test':
            out_csv = f"features_{patch}_{d_type}_nogabor_0.csv"
            df_features.to_csv(out_csv, index=False)
            df_features = pd.DataFrame(columns=col_names)
        
        elapsed = time.time() - start
        print(f"Feature extracted from {i} images. {elapsed:.2f} seconds elasped. Avg extraction time: {(elapsed)/i:.2f}")
    if d_type == 'Train':
        out_csv = f"features_{patch}_{d_type}_nogabor_0.csv"
        df_features.to_csv(out_csv, index=False)
        df_features = pd.DataFrame(columns=col_names)
        


Extracting features Naturalistic1 Test...:   0%|                                                                          | 0/50 [00:00<?, ?it/s]

AAAA (1024, 1385, 3)
BBBB (1024, 1385)
AAAA (1024, 1385, 3)
BBBB (1024, 1385)
AAAA (1024, 1385, 3)
BBBB (1024, 1385)
AAAA (1024, 1385, 3)
BBBB (1024, 1385)


C:\Users\adria\AppData\Local\Temp\ipykernel_27292\567321789.py:190: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_features = pd.concat([df_features, df], ignore_index=True)
Extracting features Naturalistic1 Test...:   2%|█▎                                                                | 1/50 [00:02<02:18,  2.82s/it]

AAAA (1024, 1385, 3)
BBBB (1024, 1385)
AAAA (1024, 1385, 3)
BBBB (1024, 1385)
AAAA (1024, 1385, 3)
BBBB (1024, 1385)


Extracting features Naturalistic1 Test...:   2%|█▎                                                                | 1/50 [00:04<03:38,  4.46s/it]


KeyboardInterrupt: 

In [45]:
df_features

,image_name,region_id,area,rgb_R_bin0,rgb_R_bin1,rgb_R_bin2,rgb_R_bin3,rgb_R_bin4,rgb_R_bin5,rgb_R_bin6,...,ASM_dist16_135deg,ASM_dist32_0deg,ASM_dist32_45deg,ASM_dist32_90deg,ASM_dist32_135deg,ASM_dist64_0deg,ASM_dist64_45deg,ASM_dist64_90deg,ASM_dist64_135deg,adversarial


In [175]:
df_features["adversarial"].value_counts()

adversarial
False    399
True     178
Name: count, dtype: int64

In [176]:
OUTPUT_CSV = "features_tjudhd_nogabor_test_inf.csv"
df_features.to_csv(OUTPUT_CSV, index=False)
print(f"✅ Saved color features to {OUTPUT_CSV} with {len(df_features)} rows")

✅ Saved color features to features_tjudhd_nogabor_test.csv with 577 rows
